In [0]:
%sql
SELECT is_account_group_member('metastore_admins');

CREATE OR REPLACE FUNCTION mrn_mask(mrn STRING)

  RETURN CASE WHEN is_member('metastore_admins')
  THEN mrn
  ELSE 'REDACTED'
  END;

ALTER TABLE silver
  ALTER COLUMN mrn
  SET MASK mrn_mask;


In [0]:
%sql
SELECT * FROM silver
ORDER BY device_id DESC;

CREATE OR REPLACE FUNCTION device_filter(devide_id INT)
 RETURN IF (is_account_group_member('admin'), true, device < 30);

 ALTER TABLE silver
 SET ROW FILTER device_filter
 ON (device_id);

In [0]:
%sql
CREATE OR REPLACE TABLE default.bronze_airlines_flights_data2 AS
SELECT * FROM read_files(
  '/Volumes/workspace/default/volume-test-1/airlines_flights_data.csv',
  format => 'csv',
  inferSchema => 'true'
);

SELECT * FROM airlines_flights_data;

In [0]:
%sql
SELECT nome, idade, RANK() OVER (PARTITION BY profissao ORDER BY idade DESC) AS rank_
FROM dbacademy_professores.professores
QUALIFY rank_ <= 3;